# Toy — partially observed relations

For each world, we generate geography, populations and a Gaussian relation matrix $R$. The generating law uses $S^*=\mathrm{strength}\,R$ and remains **fixed**: only the number of relations visible to the model changes. Training and test observations are generated once per world and reused at every coverage level.

The project is self-contained: this toy generates its own synthetic data and shares the FERM module with the Asia experiment. Each run is saved separately in `outputs/missing_relations_<profile>_<id>/`.

- `MODE='new'` runs simulations, calibration and checks.
- `MODE='load'` loads a completed run without repeating calculations.
- `PROFILE='quick'` checks the workflow with smaller budgets; `production` restores the original notebook budgets.

In [ ]:
from pathlib import Path
import sys, importlib
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'FERM new github']
            if (p / 'scripts/toy_workflow.py').is_file())
sys.path.insert(0, str(ROOT / 'scripts'))
for name in ('ferm_model', 'toy_workflow', 'toy_plots'):
    importlib.reload(importlib.import_module(name))
from toy_workflow import run_or_load, list_runs
from toy_plots import plot_run
list_runs()

## 1. Parameters

`worlds` is the number of independent worlds; `masks` is the number of masks within each world. Masks are nested and independent of relations and flows. For each origin, $\lfloor q(n-1)\rfloor$ relations are revealed, so actual coverage may differ from requested coverage.

Missing relations are set to zero without rescaling visible coefficients. Calibration selects $\theta$ by training cross-entropy; $\theta=0$ corresponds to exact RM. Missing coefficients are not reconstructed.

The `truth_particles`, `fit_particles` and `check_particles` budgets are separate. The truth is an unsmoothed finite Monte Carlo law; fitting and verification use RM-centered pseudocounts. Random streams for truth, training observations, test observations, fitting and verification are separate.

In [ ]:
MODE = 'new'            # 'new' or 'load'
RUN_NAME = None          # for 'load': toy run name returned by list_runs()
PROFILE = 'production'        # 'quick' or 'production'

CONFIG = dict(
    seed=20261016,
    worlds=30,
    nodes=40,
    masks=5,
    coverages=[0., .1, .25, .5, .75, 1.],
    strength=.7,
    sigma=1.,
    theta_grid=[0., .125, .25, .5, .75, 1., 1.5, 2.],
    train_per_origin=1000,
    test_per_origin=10000,
    truth_particles=3_000_000,
    fit_particles=20_000,
    fit_repeats=2,
    check_particles=100_000,
    check_repeats=2,
    prior=50.,
    audit_fraction=.2,
    profile=PROFILE,
)

# Quick check: retain coverages, prior sensitivity and multiple worlds/masks.
if PROFILE == 'quick':
    CONFIG.update(worlds=2, nodes=8, masks=2,
                  truth_particles=5000, fit_particles=200, check_particles=500)

# Figures: first world and first mask, fixed without selecting the best case.
WORLD = 0
MASK = 0
SHOW_COVERAGES = [0., .1, .5, 1.]
HEATMAP_CMAP = 'viridis_r'
ERROR_CMAP = 'RdBu_r'

## 2. Simulate or load

A new run saves all fitting counts, selected verification counts, truth, observations, masks and diagnostics. Predictions at the 0% and 100% endpoints are shared across masks. Prior sensitivity (5, the configured value, 500) holds $\theta$ fixed: it does not recalibrate.

Checks verify nesting, known-relation counts, shared endpoints and exclusion of the audit subset from observed relations. A run is complete only after these checks pass. In `load` mode, the saved configuration applies, not the current dictionary.

In [ ]:
run = run_or_load(mode=MODE, run_name=RUN_NAME, config=CONFIG)

## 3. Figures

The first figure compares truth and predictions using a common node ordering and shared scales. The second first averages masks within each world, then shows the mean and 10–90% percentiles **between worlds**: masks are not independent replicates.

Errors on known/hidden relations use supports that change with coverage. The audit panel instead follows the same corridors, which remain hidden through 75% coverage; they are excluded at full coverage. Comparisons are descriptive, without bootstrap or hypothesis tests and without any guarantee of monotonic improvement.

This cell reads only saved results and updates figures in the same directory without running new simulations.

In [ ]:
plot_run(run, world=WORLD, mask=MASK, coverages=SHOW_COVERAGES,
         heatmap_cmap=HEATMAP_CMAP, error_cmap=ERROR_CMAP);